# 技能2 · Day 3 上机：人机协作治理 + 组织变革

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **pandas** 分析人机协作审计日志，计算人工干预率/Agent自主完成率/人工修正率
2. 用 **matplotlib** 可视化任务完成时间分布，对比不同分工模式的效率
3. 用 **networkx** 构建组织协作网络，计算度中心性，发现桥接节点
4. 用 **McKinsey 7S 框架**评估组织AI就绪度，雷达图可视化薄弱维度
5. 用 **ADKAR 模型**诊断变革阻力，识别阻力最大的阶段
6. 用**天道推演**模拟组织变革阻力扩散路径，预判临界点

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：pandas（数据分析）+ matplotlib（可视化）+ networkx（网络分析）。
营销映射：营销团队导入AI Agent后的人机协作审计日志分析 + 组织变革评估。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> pandas / matplotlib / networkx 均为纯本地库，无需 API key，无需网络。

In [ ]:
# !pip install pandas matplotlib networkx -q

## 1. 场景背景与营销映射

**分析对象**：企业营销团队导入AI Agent后的人机协作审计日志。

**团队构成**：
- 营销策划师（人）-- 品牌战略、年度预算分配
- 文案生成Agent（AI）-- 小红书/朋友圈文案生成
- 合规审核员（人）-- 法规合规审核
- 投放优化Agent（AI）-- 广告出价、定向优化
- 数据分析师（人）-- 效果归因、用户洞察
- AI运营官（人，新角色）-- Agent监督、提示工程

**审计日志记录**：每次人机协作的完整过程，包括任务类型、执行者、是否有人工干预、干预类型、结果、耗时。

| TODO | 分析维度 | 工具 | 解决的问题 |
|------|---------|------|-----------|
| TODO1 | 审计日志统计 | pandas | 人工干预率/自主完成率/修正率 |
| TODO2 | 效率可视化 | matplotlib | 任务完成时间分布对比 |
| TODO3 | 组织网络分析 | networkx | 度中心性/桥接节点 |
| TODO4 | 组织就绪度 | McKinsey 7S | 7维度评分+雷达图 |
| TODO5 | 变革阻力诊断 | ADKAR | 5阶段评分+阻力识别 |
| TODO6 | 阻力推演 | 天道推演 | 阻力扩散路径+临界点 |

**数据说明**：审计日志基于真实文献参数（人工干预率15-30%，Stanford HAI/McKinsey报告）生成，非纯随机编造。

In [ ]:
import random
import json

# ============================================================
# 人机协作审计日志（基于真实文献参数生成）
# 参数来源：Stanford HAI AI Index / McKinsey AI状态报告
# 人工干预率15-30%为业界常见区间
# ============================================================

random.seed(42)  # 确保可复现

# 任务类型及其AI成熟度配置
# (task_type, ai_maturity, executor_dist, intervention_rate, duration_range)
TASK_CONFIG = {
    "文案生成": {
        "ai_maturity": "高",
        "executor_dist": {"Agent": 0.75, "Both": 0.20, "Human": 0.05},
        "intervention_rate": 0.18,  # AI高成熟度->低干预
        "duration_range": (30, 120),
    },
    "投放优化": {
        "ai_maturity": "高",
        "executor_dist": {"Agent": 0.80, "Both": 0.15, "Human": 0.05},
        "intervention_rate": 0.12,
        "duration_range": (10, 60),
    },
    "社媒运营": {
        "ai_maturity": "高",
        "executor_dist": {"Agent": 0.70, "Both": 0.20, "Human": 0.10},
        "intervention_rate": 0.15,
        "duration_range": (20, 90),
    },
    "用户分群": {
        "ai_maturity": "中",
        "executor_dist": {"Both": 0.50, "Agent": 0.30, "Human": 0.20},
        "intervention_rate": 0.35,
        "duration_range": (120, 600),
    },
    "竞品分析": {
        "ai_maturity": "中",
        "executor_dist": {"Both": 0.45, "Agent": 0.35, "Human": 0.20},
        "intervention_rate": 0.30,
        "duration_range": (300, 1200),
    },
    "效果归因": {
        "ai_maturity": "中",
        "executor_dist": {"Both": 0.55, "Human": 0.30, "Agent": 0.15},
        "intervention_rate": 0.40,
        "duration_range": (600, 1800),
    },
    "营销策划": {
        "ai_maturity": "低",
        "executor_dist": {"Human": 0.75, "Both": 0.20, "Agent": 0.05},
        "intervention_rate": 0.85,
        "duration_range": (1800, 3600),
    },
    "合规审核": {
        "ai_maturity": "低",
        "executor_dist": {"Human": 0.90, "Both": 0.10, "Agent": 0.00},
        "intervention_rate": 0.95,
        "duration_range": (300, 900),
    },
}

INTERVENTION_TYPES = ["修正", "驳回", "指导", "无"]
OUTCOMES = ["success", "revised", "failed"]

def generate_audit_log(n=200):
    """生成人机协作审计日志"""
    logs = []
    task_types = list(TASK_CONFIG.keys())
    base_ts = 1717200000  # 2024-06-01 base timestamp

    for i in range(n):
        task_type = random.choice(task_types)
        cfg = TASK_CONFIG[task_type]

        # 选择执行者
        executor = random.choices(
            list(cfg["executor_dist"].keys()),
            weights=list(cfg["executor_dist"].values())
        )[0]

        # 是否有人工干预
        human_intervention = random.random() < cfg["intervention_rate"]

        # 干预类型
        if human_intervention:
            intervention_type = random.choices(
                ["修正", "驳回", "指导"],
                weights=[0.50, 0.20, 0.30]
            )[0]
        else:
            intervention_type = "无"

        # 结果
        if intervention_type == "驳回":
            outcome = random.choices(OUTCOMES, weights=[0.10, 0.60, 0.30])[0]
        elif intervention_type == "修正":
            outcome = random.choices(OUTCOMES, weights=[0.30, 0.65, 0.05])[0]
        else:
            outcome = random.choices(OUTCOMES, weights=[0.88, 0.10, 0.02])[0]

        # 耗时
        dur_min, dur_max = cfg["duration_range"]
        if executor == "Agent" and not human_intervention:
            duration = random.uniform(dur_min, dur_min + (dur_max - dur_min) * 0.3)
        elif executor == "Both":
            duration = random.uniform(dur_min, dur_max)
        else:
            duration = random.uniform(dur_min + (dur_max - dur_min) * 0.5, dur_max)

        timestamp = base_ts + i * 1800 + random.randint(0, 600)

        logs.append({
            "task_id": f"T{i+1:03d}",
            "task_type": task_type,
            "executor": executor,
            "agent_action": f"Agent执行{task_type}任务" if executor != "Human" else "无(Agent未参与)",
            "human_intervention": human_intervention,
            "intervention_type": intervention_type,
            "outcome": outcome,
            "duration_sec": round(duration, 1),
            "timestamp": timestamp,
        })

    return logs

# 生成审计日志
AUDIT_LOGS = generate_audit_log(200)

# 组织协作网络数据（基于McKinsey Agentic Organization模型）
# 节点 = 角色（人/Agent），边 = 协作关系
ORG_EDGES = [
    ("营销策划师", "文案生成Agent", 0.8),      # 策划师指导Agent
    ("文案生成Agent", "合规审核员", 0.7),        # Agent产出->人审核
    ("合规审核员", "营销策划师", 0.6),           # 审核反馈->策划
    ("营销策划师", "投放优化Agent", 0.5),       # 策划->投放指导
    ("投放优化Agent", "数据分析师", 0.7),       # 投放数据->分析
    ("数据分析师", "营销策划师", 0.8),           # 分析->策划反馈
    ("AI运营官", "文案生成Agent", 0.9),         # AI运营官监督Agent
    ("AI运营官", "投放优化Agent", 0.9),         # AI运营官监督Agent
    ("AI运营官", "营销策划师", 0.5),            # AI运营官<->策划师
    ("合规审核员", "AI运营官", 0.4),            # 审核<->AI运营
    ("数据分析师", "AI运营官", 0.5),            # 分析<->AI运营
    ("文案生成Agent", "投放优化Agent", 0.3),    # Agent间协作
    ("营销策划师", "数据分析师", 0.6),          # 策划<->分析
]

# McKinsey 7S 评估数据（1-5分，5分最佳）
SEVEN_S_SCORES = {
    "Strategy": 4,       # AI战略较清晰
    "Structure": 2,      # 组织结构未适配Agent（薄弱）
    "Systems": 3,        # 系统部分就绪
    "Shared Values": 3,  # 价值观转型中
    "Skills": 2,         # AI技能缺口大（薄弱）
    "Style": 3,          # 管理风格转型中
    "Staff": 4,          # 已招聘AI运营官
}

# ADKAR 变革阻力评估数据（1-5分，5分阻力最小/准备度最高）
ADKAR_SCORES = {
    "Awareness": 4,    # 员工已认知AI变革必要性
    "Desire": 2,       # 员工变革意愿低（阻力最大）
    "Knowledge": 3,    # 部分员工已接受AI培训
    "Ability": 2,      # 实操能力不足（阻力大）
    "Reinforcement": 3, # 巩固机制部分建立
}

# 组织成员变革阻力模型（用于天道推演）
# role: (初始阻力值0-1, 影响力0-1, 连接数)
ORG_MEMBERS = {
    "营销策划师": {"resistance": 0.3, "influence": 0.8, "connections": 3},
    "文案生成Agent": {"resistance": 0.0, "influence": 0.5, "connections": 3},
    "合规审核员": {"resistance": 0.6, "influence": 0.6, "connections": 3},
    "投放优化Agent": {"resistance": 0.0, "influence": 0.5, "connections": 3},
    "数据分析师": {"resistance": 0.4, "influence": 0.7, "connections": 4},
    "AI运营官": {"resistance": 0.1, "influence": 0.7, "connections": 4},
}

print("环境初始化完成")
print(f"审计日志: {len(AUDIT_LOGS)} 条记录")
print(f"组织网络: {len(ORG_EDGES)} 条协作关系")
print(f"7S评估: {len(SEVEN_S_SCORES)} 个维度")
print(f"ADKAR评估: {len(ADKAR_SCORES)} 个阶段")

## TODO 1：用 pandas 分析审计日志 -- 人工干预率/自主完成率/修正率

**pandas** 是 Python 数据分析事实标准。核心 API：
- `pd.DataFrame(data)`：从列表创建 DataFrame
- `df.groupby("col").agg(...)`：分组聚合
- `df["col"].value_counts()`：频率统计
- `df["col"].mean()`：均值

**关键指标定义**：
- 人工干预率 = 有干预的记录数 / 总记录数
- Agent自主完成率 = (executor=="Agent" 且 无干预) 的记录数 / 总记录数
- 人工修正率 = intervention_type=="修正" 的记录数 / 总记录数

**业界基准**（Stanford HAI / McKinsey）：人工干预率 15-30% 为业界常见区间。

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 用 pandas 将 AUDIT_LOGS 转为 DataFrame
#       2) 计算人工干预率、Agent自主完成率、人工修正率
#       3) 按 task_type 分组计算各任务类型的干预率
# 提示: import pandas as pd
#       df = pd.DataFrame(AUDIT_LOGS)
#       intervention_rate = df["human_intervention"].mean()
#       autonomous_rate = ((df["executor"]=="Agent") & (~df["human_intervention"])).mean()
#       correction_rate = (df["intervention_type"]=="修正").mean()
#       by_task = df.groupby("task_type")["human_intervention"].mean()
audit_report = None  # TODO: 计算审计指标
raise NotImplementedError("你的代码")
# ====================

print(f"审计报告: {audit_report}")

## TODO 2：用 matplotlib 可视化任务完成时间分布

**matplotlib** 是 Python 可视化基础库。核心 API：
- `plt.figure(figsize=(w,h))`：创建画布
- `plt.boxplot(data, labels=...)`：箱线图对比分布
- `plt.bar(x, y)`：柱状图
- `plt.title/plt.xlabel/plt.ylabel`：标签

**分析目标**：
1. 箱线图：对比 Agent / Human / Both 三种执行者的任务耗时分布
2. 柱状图：各任务类型的人工干预率对比

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 箱线图：对比 Agent/Human/Both 三种执行者的任务耗时分布
#       2) 柱状图：各任务类型的人工干预率
# 提示: import matplotlib.pyplot as plt
#       fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#       axes[0].boxplot([df[df["executor"]==e]["duration_sec"] for e in ["Agent","Human","Both"]], labels=...)
#       axes[1].bar(by_task.index, by_task.values)
raise NotImplementedError("你的代码")
# ====================

plt.tight_layout()
plt.savefig("task_analysis.png", dpi=100, bbox_inches="tight")
plt.show()
print("图表已保存为 task_analysis.png")

## 2. 组织协作网络分析

当 Agent 成为组织一等成员（Agentic Organization），组织结构从"树形"变为"网络"。networkx 可以量化分析这种网络结构。

**关键概念**：
- **度中心性（Degree Centrality）**：节点的连接数占比，值越高=协作枢纽
- **桥接中心性（Betweenness Centrality）**：节点处于最短路径上的频率，值越高=信息瓶颈
- **桥接节点**：连接不同子群的节点，移除后网络可能断裂

## TODO 3：用 networkx 构建组织协作网络

**networkx** 核心 API：
- `nx.Graph()`：创建无向图
- `G.add_node(name)` / `G.add_edge(a, b, weight=w)`：添加节点/边
- `nx.degree_centrality(G)`：度中心性
- `nx.betweenness_centrality(G)`：桥接中心性
- `nx.draw(G, with_labels=True)`：可视化网络

**分析目标**：识别组织中的协作枢纽和信息瓶颈（桥接节点）。

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 用 networkx 构建组织协作网络（节点=角色，边=协作关系）
#       2) 计算度中心性和桥接中心性
#       3) 识别桥接节点（betweenness最高的节点）
#       4) 可视化网络
# 提示: import networkx as nx
#       G = nx.Graph()
#       for a, b, w in ORG_EDGES: G.add_edge(a, b, weight=w)
#       deg = nx.degree_centrality(G)
#       btw = nx.betweenness_centrality(G, weight="weight")
#       nx.draw(G, with_labels=True, node_color="lightblue", node_size=2000)
network_report = None  # TODO: 网络分析报告
raise NotImplementedError("你的代码")
# ====================

print(f"网络报告: {network_report}")

## TODO 3 续：用 McKinsey 7S 框架评估组织AI就绪度

**McKinsey 7S 框架**：7个维度评估组织一致性（1-5分，5分最佳）。
- Strategy（战略）/ Structure（结构）/ Systems（系统）
- Shared Values（共同价值观）/ Skills（技能）/ Style（风格）/ Staff（人员）

**分析目标**：用雷达图可视化7S评分，识别薄弱维度（分数最低的维度是AI导入的瓶颈）。

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 用 McKinsey 7S 数据画雷达图
#       2) 识别薄弱维度（分数最低的2个）
# 提示: import numpy as np
#       categories = list(SEVEN_S_SCORES.keys())
#       values = list(SEVEN_S_SCORES.values())
#       angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
#       fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))
#       ax.plot(angles + angles[:1], values + values[:1])
#       ax.fill(angles, values, alpha=0.3)
seven_s_report = None  # TODO: 7S评估报告
raise NotImplementedError("你的代码")
# ====================

print(f"7S评估报告: {seven_s_report}")

## TODO 4：用 ADKAR 模型诊断变革阻力

**ADKAR 模型**：5个阶段的变革准备度评估（1-5分，5分阻力最小）。
- Awareness（认知）/ Desire（意愿）/ Knowledge（知识）
- Ability（能力）/ Reinforcement（巩固）

**分析目标**：识别阻力最大的阶段（分数最低），设计针对性干预。

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 用 ADKAR 数据画柱状图
#       2) 识别阻力最大的阶段（分数最低）
#       3) 设计针对性干预建议
# 提示: stages = list(ADKAR_SCORES.keys())
#       scores = list(ADKAR_SCORES.values())
#       plt.bar(stages, scores, color=["red" if s<3 else "green" for s in scores])
#       weakest = min(ADKAR_SCORES, key=ADKAR_SCORES.get)
adkar_report = None  # TODO: ADKAR诊断报告
raise NotImplementedError("你的代码")
# ====================

print(f"ADKAR诊断报告: {adkar_report}")

## TODO 5：用天道推演模拟组织变革阻力扩散路径

**天道推演**：以天神视角俯视组织变革，模拟不同干预策略下阻力的演化路径。

**阻力扩散模型**：
- 每个成员有初始阻力值（0-1）和影响力（0-1）
- 阻力通过协作网络扩散：高阻力成员影响连接的低阻力成员
- 临界点：当平均阻力超过阈值（如0.5），变革可能失败

**推演步骤**：
1. 初始化各成员阻力值
2. 模拟N轮扩散（每轮：阻力通过边传播）
3. 记录每轮平均阻力，识别临界点
4. 对比"无干预"vs"干预AI运营官降阻力"两个场景

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 初始化各成员阻力值
#       2) 模拟N轮阻力扩散（每轮：阻力通过边传播）
#       3) 记录每轮平均阻力，识别临界点
#       4) 对比"无干预"vs"干预"两个场景
# 提示: members = dict(ORG_MEMBERS)  # 复制初始状态
#       for epoch in range(N_EPOCHS):
#           for member in members:
#               neighbors = [n for n in G.neighbors(member)]
#               diffusion = sum(members[n]["resistance"]*members[n]["influence"] for n in neighbors) / len(neighbors)
#               members[member]["resistance"] = min(1.0, members[member]["resistance"]*0.9 + diffusion*0.1)
#           avg_resistance = np.mean([m["resistance"] for m in members.values()])
simulation_report = None  # TODO: 天道推演报告
raise NotImplementedError("你的代码")
# ====================

print(f"天道推演报告: {simulation_report}")

## 3. 反思与前沿

### 反思问题
1. 审计日志显示哪类任务的人工干预率最高？根因是什么？（AI成熟度不足？任务复杂度过高？治理流程问题？）
2. 组织网络中的桥接节点是谁？如果这个人离职，网络会怎样？
3. McKinsey 7S 中哪个维度最薄弱？这对AI导入意味着什么？
4. ADKAR 中 Desire（意愿）分数最低，说明什么？如何提升？
5. 天道推演显示阻力扩散的临界点在第几轮？如何提前干预？

### 2026 前沿：Agentic Organization + Computer Use 审计
- **Agentic Organization**（McKinsey）：Agent成为组织一等成员，重塑工作定义/组织结构/治理体系
- **Computer Use / 计算机使用**：Agent直接操作GUI，审计日志需记录每步GUI操作（鼠标/键盘/截图）
- **天道推演×组织变革**：用多Agent仿真模拟组织成员群体动力学，预判阻力扩散路径
- **多Agent仿真**：将组织成员建模为Agent，模拟协作/冲突/博弈，预判组织变革结果

参考 [McKinsey Agentic Organization](https://www.mckinsey.com/capabilities/mckinsey-digital/our-insights/the-economic-potential-of-generative-ai) + [Stanford HAI AI Index](https://aiindex.stanford.edu/report/) + [Anthropic Computer Use](https://docs.anthropic.com/en/docs/build-with-claude/computer-use)。